# Signal Library

Construct all 10 base signals using synthetic data from `src/data_bridge.py`.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from pathlib import Path

from src.data_bridge import load_all_signal_inputs
from src.aqr_data import _fallback_factors
from src.signals import (
    signal_sue, signal_earnings_revision, signal_momentum,
    signal_reversal, signal_earnings_yield, signal_roe_stability,
    signal_accruals, signal_realized_vol, signal_factor_loading,
    build_signal_panel,
)

## Load inputs

In [ ]:
data = load_all_signal_inputs(n_tickers=100, start="2012-01-01", end="2024-12-31")

sue_df       = data["sue_df"]
prices       = data["prices"]
fundamentals = data["fundamentals"]
accruals_df  = data["accruals"]
earnings_df  = data["earnings_estimates"]

print(f"Tickers:          {prices['ticker'].nunique()}")
print(f"Price date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
print(f"Quarter dates:    {sue_df['date'].nunique()} quarters")

## Precompute monthly stock returns (needed for signals 9 & 10)

In [ ]:
wide_prices  = prices.pivot(index="date", columns="ticker", values="adj_close")
monthly      = wide_prices.resample("ME").last()
stock_returns = monthly.pct_change().dropna(how="all")

factors = _fallback_factors()
hml = factors["HML"]
umd = factors["UMD"]

print(f"Stock returns: {stock_returns.shape}  ({stock_returns.index[0].date()} to {stock_returns.index[-1].date()})")
print(f"Factor data:   {factors.shape}  ({factors.index[0].date()} to {factors.index[-1].date()})")

## Construct all 10 signals

Each constructor is wrapped in try/except. Failures print a warning and are skipped; the panel is built from whatever succeeds.

In [ ]:
def describe(name, df):
    print(f"  {name:<30}  obs={len(df):>7,}  "
          f"{df['date'].min().date()} to {df['date'].max().date()}  "
          f"tickers={df['ticker'].nunique()}")

def try_signal(name, fn, *args, **kwargs):
    try:
        df = fn(*args, **kwargs)
        describe(name, df)
        return df
    except Exception as e:
        print(f"  WARNING: {name} failed — {e}")
        return None

candidates = {
    "sue":      lambda: try_signal("1. SUE",                   signal_sue,              sue_df),
    "revision": lambda: try_signal("2. Earnings Revision",     signal_earnings_revision, earnings_df),
    "momentum": lambda: try_signal("3. Momentum (12-1)",       signal_momentum,         prices),
    "reversal": lambda: try_signal("4. Short-Term Reversal",   signal_reversal,         prices),
    "ep":       lambda: try_signal("5. Earnings Yield (E/P)",  signal_earnings_yield,   fundamentals, prices),
    "roe_stab": lambda: try_signal("6. ROE Stability",         signal_roe_stability,    fundamentals),
    "accruals": lambda: try_signal("7. Accruals (Sloan)",      signal_accruals,         accruals_df),
    "rvol":     lambda: try_signal("8. Realized Volatility",   signal_realized_vol,     prices),
    "hml_beta": lambda: try_signal("9. AQR Value (HML beta)",  signal_factor_loading,   stock_returns, hml, 60),
    "umd_beta": lambda: try_signal("10. AQR Momentum (UMD beta)", signal_factor_loading, stock_returns, umd, 60),
}

signals = {}
for key, build in candidates.items():
    result = build()
    if result is not None:
        signals[key] = result

print(f"\nSuccessfully built {len(signals)}/10 signals: {list(signals.keys())}")

## Merge into signal panel

In [ ]:
panel = build_signal_panel(signals)
print(f"Panel shape: {panel.shape}")
print(f"\nNon-NaN observations per signal:")
sig_cols = [c for c in panel.columns if c not in ("ticker", "date")]
print(panel[sig_cols].notna().sum().to_string())

## Save panel

In [ ]:
out_path = Path("../data/processed/signal_panel.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
panel.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")